In [221]:
import pandas as pd
file_path ='MRNGO_features.xlsx'

df1 = pd.read_excel(file_path, sheet_name='features')
df2 = pd.read_excel(file_path, sheet_name='concept_features')

merged_df = pd.merge(df1, df2, on='Feature', how='left')

file_path = 'MRNGO_features_clustered_stimulimaking_preprocessed.xlsx'
df3 = pd.read_excel(file_path)

df_final = pd.merge(merged_df, df3, left_on='Feature', right_on='vector_lemma_C', how='inner')

cols = list(df_final.columns)
cols.insert(cols.index('Feature') + 1, cols.pop(cols.index('vector_lemma_C')))
df_final = df_final[cols]

cols_to_drop = set()
for i in range(len(df_final.columns)):
    col_name_1 = df_final.columns[i]
    if col_name_1 in cols_to_drop:
        continue
    for j in range(i + 1, len (df_final.columns)):
        col_name_2 = df_final.columns[j]
        if df_final.iloc[:, i].equals(df_final.iloc[:, j]):
            cols_to_drop.add(col_name_2)
df_final = df_final.drop(columns=cols_to_drop)

df_final.to_excel('pilot_features.xlsx', sheet_name='sorted_by_features', index=False)


In [2]:
import pandas as pd
import numpy as np
import os 

file_path = 'MRNGO_features_pilot.xlsx'
sheet_name = 'sorted_by_features'
column_to_check = 'itemno'
new_column_name = 'filter_status'

# features that we did not visualized at the end 
items_to_exclude = [ 
    'f00014',
    'f00213',
    'f00217',
    'f00220',
    'f00486',
    'f00497',
    'f00501',
    'f00649',
    'f00991',
    'f01105',
    'f01406',
    'f01528',
    'f01932',
    'f02259'
]

if not os.path.exists(file_path):
    print(f"file was not find")
else:
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name)
        df[new_column_name] = np.where(
            df[column_to_check].isin(items_to_exclude),
            'excluded',
            'included'
        )

        with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            df.to_excel(writer,sheet_name=sheet_name, index=False)
        print(f"success, sheet has been updated")

    except PermissionError:
        print(f"permission error")

success, sheet has been updated


In [4]:
import pandas as pd
file_path = 'MRNGO_features_pilot.xlsx'
sheet_name = 'sorted_by_features'
sorted_feature_df_main = df[(df['filter_status'] == 'included') & (df['Feature'] != 'HOL_BOLT')]

# filtering out features that were excluded from the set and also the feature 'HOL_BOLT'


In [5]:
import numpy as np

def proportional_10(col, which='top'):
    asc = (which == 'bottom')

    def pick(g):
        vals = sorted(g[col].dropna().unique(), reverse=not asc)
        vals = vals[:10]                              # if >10 distinct values, keep most extreme 10
        sub = g[g[col].isin(vals)]
        cnt = sub[col].value_counts().reindex(vals, fill_value=0)

        if len(vals) == 10:
            n = pd.Series(1, index=vals)             # 1 from each
        else:
            q = cnt / cnt.sum() * 10
            n = np.floor(q).astype(int)
            n[n == 0] = 1                            # include each value
            while n.sum() > 10: n[n.idxmax()] -= 1
            while n.sum() < 10: n[(q - n).idxmax()] += 1

        return pd.concat([sub[sub[col].eq(v)].sample(n=min(n[v], len(sub[sub[col].eq(v)])), random_state=42) for v in vals])

    return sorted_feature_df_main.groupby('Concept', group_keys=False).apply(pick).reset_index(drop=True)



In [6]:
df_freq_x_top10 = df.groupby('Concept').apply(
    lambda x: x.nlargest(10, 'frequency_x')
).reset_index(drop=True)
list_sequence_1 = df_freq_x_top10.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_1)

   Conceptno                                            Feature
0        a05  [RÉSZE_LÁB, ILYEN_BARNA, FAJTA_ÁLLAT, RÉSZE_FA...
1        a16  [RÉSZE_LÁB, ILYEN_NAGY, ILYEN_BARNA, HOL_HÁZ, ...
2        a19  [ILYEN_NAGY, ILYEN_BARNA, FAJTA_ÁLLAT, RÉSZE_F...
3        b02  [ILYEN_BARNA, LEHET_OLVAS, RÉSZE_BELSŐ, ALFAJT...
4        b10  [LEHET_ESZIK, RÉSZE_UJJ, LEHET_ÍR, LEHET_FELVE...
5        b11  [RÉSZE_UJJ, ILYEN_HOSSZÚ, HOL_EMBER, RÉSZE_TAL...
6        c10  [ILYEN_NAGY, ILYEN_HOSSZÚ, LEHET_FELVESZ, HOL_...
7        c13  [RÉSZE_BELSŐ, ILYEN_HOSSZÚ, LEHET_FELVESZ, HOL...
8        c28  [LEHET_FELVESZ, HOL_EMBER, KI_EMBER, ILYEN_PUH...
9        f12  [LEHET_ESZIK, FAJTA_GYÜMÖLCS, ILYEN_PIROS, RÉS...
10       f15  [LEHET_ESZIK, FAJTA_GYÜMÖLCS, ILYEN_SÁRGA, ALF...
11       f17  [LEHET_ESZIK, FAJTA_GYÜMÖLCS, ILYEN_PIROS, RÉS...
12       l02  [ILYEN_NAGY, HOL_HÁZ, LEHET_MEGY, HOL_FÖLD, HO...
13       l12  [HOL_HÁZ, LEHET_JÁTSZIK, BENNE_EMBER, BENNE_ES...
14       l23  [ILYEN_NAGY, ILYEN_BARNA, 

In [11]:
df_freq_x_low10 = df.groupby('Concept').apply(
    lambda x: x.nsmallest(10, 'frequency_x')
).reset_index(drop=True)
list_sequence_2 = df_freq_x_low10.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_2)

   Conceptno                                            Feature
0        a05  [HOL_ÁLLATKERT, ILYEN_ALACSONY, HOL_MESE, ILYE...
1        a16  [ILYEN_BOLDOG, ILYEN_HANGOS, RÉSZE_HAS, TUD_SÉ...
2        a19  [HOL_ÁLLATKERT, ILYEN_BOLDOG, RÉSZE_HAS, TUD_S...
3        b02  [LEHET_NÉZ, RÉSZE_KÖR, TUD_SEGÍT, HOL_MESE, IL...
4        b10  [LEHET_MOZOG, RÉSZE_HÚS, RÉSZE_BŐR, RÉSZE_KÖR,...
5        b11  [HOL_TÉL, LEHET_MOZOG, RÉSZE_BOKA, EMBER_KI, L...
6        c10  [ILYEN_NÉGYZETALAKÚ, NEMILYEN_KOSZOS, RÉSZE_BO...
7        c13  [HOL_KINT, NEMILYEN_KOSZOS, AKCIÓ_VÍZ, MIBŐL_F...
8        c28  [HOL_TÉL, LEHET_MOZOG, RÉSZE_HAS, ILYEN_SZÍNES...
9        f12  [HOL_HŰTŐ, ILYEN_SAVANYKÁS, MÓD_MEGMOS_FOGYASZ...
10       f15  [HOL_HŰTŐ, ILYEN_SAVANYKÁS, BENNE_MAG, HOL_BOK...
11       f17  [ILYEN_SAVANYKÁS, MÓD_MEGMOS_FOGYASZT, BENNE_M...
12       l02  [HOL_KINT, HOL_ÁLLATKERT, ILYEN_PICI, TUD_ÜL, ...
13       l12  [SOK_ABLAK, LEHET_NÉZ, FAJTA_HELY, HOL_SZOBA, ...
14       l23  [BENNE_FÖLD, ILYEN_ALACSON

In [7]:
df_freq_y_top10 = proportional_10('frequency_y', 'top') 
list_sequence_3 = df_freq_y_top10.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_3)

   Conceptno                                            Feature
0        a05  [FAJTA_ÁLLAT, ILYEN_SÁRGA, ILYEN_BARNA, RÉSZE_...
1        a16  [RÉSZE_FAROK, RÉSZE_LÁB, FAJTA_ÁLLAT, NÉGY_LÁB...
2        a19  [FAJTA_ÁLLAT, RÉSZE_FAROK, HOL_FARM, NÉGY_LÁB,...
3        b02  [ILYEN_BARNA, ALFAJTA_ZÖLD, EMBER_RÉSZE, HOL_E...
4        b10  [RÉSZE_UJJ, RÉSZE_KÖRÖM, EMBER_RÉSZE, LEHET_FO...
5        b11  [LEHET_JÁR, FAJTA_TESTRÉSZ, RÉSZE_TÉRD, HOL_TE...
6        c10  [LEHET_FELVESZ, ILYEN_NAGY, ILYEN_HOSSZÚ, HOL_...
7        c13  [LEHET_FELVESZ, RÉSZE_TALP, RÉSZE_ORR, HOL_EMB...
8        c28  [LEHET_FELVESZ, RÉSZE_MINTA, FAJTA_RUHADARAB, ...
9        f12  [ILYEN_PIROS, ALFAJTA_ZÖLD, FAJTA_GYÜMÖLCS, HO...
10       f15  [ILYEN_SAVANYÚ, ILYEN_SÁRGA, LEHET_ESZIK, RÉSZ...
11       f17  [RÉSZE_MAG, ILYEN_PIROS, LEHET_ESZIK, ILYEN_FI...
12       l02  [BENNE_JÁTSZIK, BENNE_JÁTÉK, BENNE_BARÁT, BENN...
13       l12  [HOL_HÁZ, LEHET_JÁTSZIK, BENNE_EMBER, HOL_LAKÁ...
14       l23  [FAJTA_HELY, BENNE_EMBER, 

In [8]:
df_freq_y_low10 = proportional_10('frequency_y', 'bottom')  
list_sequence_4 = df_freq_y_low10.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_4)

   Conceptno                                            Feature
0        a05  [ALFAJTA_ZÖLD, HOL_ÁLLATKERT, ILYEN_ALACSONY, ...
1        a16  [ILYEN_HOSSZÚ, HOL_KERT, TUD_JÁR, ILYEN_NAGY, ...
2        a19  [TUD_SZÁLLÍT, HOL_ÁLLATKERT, ILYEN_GYORS, ILYE...
3        b02  [ILYEN_SOKSZÍNŰ, LEHET_NÉZ, ILYEN_FEHÉR, LEHET...
4        b10  [LEHET_FELVESZ, TUD_RAJZOL, LEHET_VEZET, LEHET...
5        b11  [RÉSZE_KÖRÖM, LEHET_MOZOG, RÉSZE_TALP, RÉSZE_S...
6        c10  [ILYEN_FEKETE, KI_GYEREK, LEHET_VÁG, MIBŐL_FON...
7        c13  [LEHET_VESZ, MIBŐL_CÉRNA, ILYEN_HOSSZÚ, NEMILY...
8        c28  [HOL_TEST, ILYEN_SZÍNES , LEHET_VESZ, RÉSZE_KÉ...
9        f12  [HOL_ISKOLA, RÉSZE_BELSŐ, RÉSZE_LEVÉL, RÉSZE_M...
10       f15  [ALFAJTA_ZÖLD, LEHET_SZED, TUD_TEREM, HOL_ÉTTE...
11       f17  [ILYEN_FEHÉR, RÉSZE_PÖTTY, ILYEN_SAVANYKÁS, HO...
12       l02  [HOL_FÖLD, KI_GYEREK, TUD_JÁTSZIK, HOL_VÁROS, ...
13       l12  [FAJTA_HELY, BENNE_JÁTÉK, KI_GYEREK, BENNE_BAR...
14       l23  [ILYEN_SZÉP, LEHET_VEZET, 

In [9]:
df_dist_top10 = proportional_10('Distinct', 'top')
list_sequence_5 = df_dist_top10.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_5)

   Conceptno                                            Feature
0        a05  [ILYEN_CUKI, RÉSZE_FAROK, RÉSZE_SZEM, TUD_MEGY...
1        a16  [RÉSZE_FAROK, TUD_FUT, TUD_JÁTSZIK, HOL_VÁROS,...
2        a19  [RÉSZE_FAROK, RÉSZE_FÜL, RÉSZE_NYAK, RÉSZE_HAN...
3        b02  [LEHET_NÉZ, EMBER_RÉSZE, ILYEN_FONTOS, ILYEN_K...
4        b10  [LEHET_FOG, RÉSZE_BŐR, ILYEN_FONTOS, HOL_TEST,...
5        b11  [RÉSZE_LÁBFEJ, RÉSZE_COMB, HOL_TEST, RÉSZE_KÖR...
6        c10  [RÉSZE_COMB, LEHET_HÚZ, HOL_TEST, ILYEN_VASTAG...
7        c13  [NEMILYEN_KOSZOS, FAJTA_RUHADARAB, RÉSZE_TALP,...
8        c28  [RÉSZE_HAS, FAJTA_RUHADARAB, RÉSZE_KÉZ, ILYEN_...
9        f12  [HOL_HŰTŐ, FAJTA_GYÜMÖLCS, ILYEN_FINOM, RÉSZE_...
10       f15  [LEHET_MEGKÓSTOL, RÉSZE_MAG, HOL_HŰTŐ, FAJTA_G...
11       f17  [ILYEN_SAVANYKÁS, MÓD_MEGMOS_FOGYASZT, RÉSZE_P...
12       l02  [BENNE_BARÁT, HOL_KINT, TUD_ÜL, BENNE_EMBER, H...
13       l12  [FAJTA_HELY, BENNE_JÁTÉK, SOK_ABLAK, BENNE_ESZ...
14       l23  [ILYEN_SZÉP, BENNE_ÍRÁS, L

In [10]:
df_dist_low10 = proportional_10('Distinct', 'bottom')   # or 'bottom'
list_sequence_6 = df_dist_low10.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_6)

   Conceptno                                            Feature
0        a05  [ALFAJTA_KICSI, ILYEN_FEHÉR, RÉSZE_LÁB, FAJTA_...
1        a16  [ILYEN_NAGY, HOL_HÁZ, ILYEN_HOSSZÚ, LEHET_JÁTS...
2        a19  [ILYEN_NAGY, ILYEN_BARNA, ILYEN_FEHÉR, ILYEN_G...
3        b02  [ILYEN_BARNA, RÉSZE_BELSŐ, KI_EMBER, ILYEN_FEH...
4        b10  [LEHET_JÁTSZIK, LEHET_ESZIK, MIBŐL_BŐR, HOL_EM...
5        b11  [ILYEN_HOSSZÚ, KI_EMBER, MIBŐL_BŐR, HOL_EMBER,...
6        c10  [ILYEN_NAGY, ILYEN_HOSSZÚ, LEHET_VESZ, HOL_UTC...
7        c13  [HOL_PIAC, LEHET_VESZ, HOL_UTCA, FAJTA_TÁRGY, ...
8        c28  [HOL_OTTHON, LEHET_VESZ, LEHET_FELVESZ, HOL_EM...
9        f12  [HOL_PIAC, ILYEN_KICSI, ALFAJTA_ZÖLD, ILYEN_PI...
10       f15  [RÉSZE_BELSŐ, ALFAJTA_ZÖLD, LEHET_ESZIK, ILYEN...
11       f17  [RÉSZE_BELSŐ, LEHET_VESZ, ILYEN_FEHÉR, ILYEN_P...
12       l02  [ILYEN_NAGY, HOL_HÁZ, HOL_FÖLD, LEHET_MEGY, KI...
13       l12  [HOL_HÁZ, LEHET_JÁTSZIK, KI_GYEREK, BENNE_EMBE...
14       l23  [ILYEN_NAGY, ILYEN_BARNA, 

In [11]:
df_random = sorted_feature_df_main.groupby('Concept').apply(
    lambda x: x.sample(n=min(len(x), 10))
).reset_index(drop=True)
list_sequence_7 = df_random.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_7)

   Conceptno                                            Feature
0        a05  [RÉSZE_SZEM, HOL_MESE, RÉSZE_FAROK, ILYEN_KÉK,...
1        a16  [ILYEN_HANGOS, HOL_ERDŐ, HOL_UTCA, ILYEN_PUHA,...
2        a19  [ILYEN_FEKETE, RÉSZE_HAS, ILYEN_BARNA, LEHET_L...
3        b02  [ILYEN_FONTOS, HOL_EMBER, RÉSZE_SZÍN, RÉSZE_BE...
4        b10  [LEHET_FOG, KI_EMBER, TUD_RAJZOL, EMBER_RÉSZE,...
5        b11  [EMBER_KI, RÉSZE_LÁBFEJ, KI_EMBER, FAJTA_TESTR...
6        c10  [MIBŐL_SZÖVET, ILYEN_NAGY, ILYEN_FEKETE, HOL_T...
7        c13  [FAJTA_RUHADARAB, MIVEL_VARR, RÉSZE_KÜLSŐ, RÉS...
8        c28  [RÉSZE_HAS, MIBŐL_CÉRNA, RÉSZE_MINTA, ILYEN_PU...
9        f12  [RÉSZE_BELSŐ, RÉSZE_SZÁR, ILYEN_SÁRGA, ILYEN_E...
10       f15  [ILYEN_EGÉSZSÉGES, ILYEN_SAVANYÚ, FAJTA_GYÜMÖL...
11       f17  [RÉSZE_MAG, TUD_TEREM, LEHET_VESZ, RÉSZE_BELSŐ...
12       l02  [BENNE_EMBER, ILYEN_SZÍNES, BENNE_JÁTSZIK, HOL...
13       l12  [FAJTA_HELY, KI_GYEREK, HOL_LAKÁS, LEHET_JÁTSZ...
14       l23  [LEHET_SÉTÁL, BENNE_BARÁT,

In [12]:
from pathlib import Path
import pandas as pd

out_root = Path("sequences")
feat_map = (sorted_feature_df_main[["Feature", "itemno"]]
            .drop_duplicates("Feature")
            .set_index("Feature")["itemno"])

versions = ["A", "B", "C", "D", "E"]
data_frames = [list_sequence_3, list_sequence_4, list_sequence_5, list_sequence_6, list_sequence_7]

for version, df in zip(versions, data_frames):
    for rowno, row in df.iterrows():
        concept = row["Conceptno"]
        features = row["Feature"]

        concept_dir = out_root / concept
        concept_dir.mkdir(parents=True, exist_ok=True)

        out_df = pd.DataFrame({
            "trialno": [f"{i:02d}" for i in range(1, len(features) + 1)],
            "feature_name": features,
        })
        out_df["feature_file"] = out_df["feature_name"].map(feat_map).map(lambda x: f"stimuli/{x}.png")

        out_df.to_csv(concept_dir / f"{concept}_list{version}.csv", index=False, encoding="utf-8-sig")

In [33]:
from pathlib import Path
import pandas as pd

rng = 42

base = sorted_feature_df_main[["Conceptno", "Concept"]].drop_duplicates("Conceptno").copy()
base["initial"] = base["Conceptno"].str[0]

if base["initial"].nunique() > 10:
    raise ValueError("More than 10 unique first characters, so all cannot be included in a 10-item demo list.")

must_have = base.groupby("initial", group_keys=False).apply(lambda g: g.sample(1, random_state=rng))
rest = base.drop(must_have.index).sample(n=10 - len(must_have), random_state=rng)

demo = pd.concat([must_have, rest]).sample(frac=1, random_state=rng).reset_index(drop=True)

versions = pd.Series(["A","A","B","B","C","C","D","D","E","E"]).sample(frac=1, random_state=rng).reset_index(drop=True)

demo.insert(0, "concept_trial", [f"{i:02d}" for i in range(1, 11)])
demo.insert(1, "version", versions)
demo = demo[["concept_trial", "version", "Conceptno", "Concept"]]
demo.columns = ["concept_trial", "version", "conceptno", "concept"]

Path("concept_lists").mkdir(exist_ok=True)
demo.to_csv("concept_lists/demo.csv", index=False, encoding="utf-8-sig")